# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [ ]:
# %pip install mlflow

In [1]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [2]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

🏃 View run omniscient-seal-986 at: http://localhost:5000/#/experiments/1/runs/8be4dd76e0ab49c9a8c3375c7569c8a7
🧪 View experiment at: http://localhost:5000/#/experiments/1


Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [4]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2026/05/26 12:31:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/26 12:31:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'sk-learn-random-forest-reg-model'.
2026/05/26 12:31:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 1


🏃 View run judicious-fly-530 at: http://localhost:5000/#/experiments/1/runs/d1a824ebea1848109b91e657074cbb80
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '1' of model 'sk-learn-random-forest-reg-model'.


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [5]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.49980732  44.5541425   41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.44675489 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [6]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [ ]:
# %pip install hyperopt
# %pip install -U git+https://github.com/hyperopt/hyperopt

In [9]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [10]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [11]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

2026/05/26 12:35:40 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779791740229, experiment_id='2', last_update_time=1779791740229, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>

In [12]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

  0%|          | 0/8 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:From c:\Users\urkow\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.

Epoch 1/3                                            

  0%|          | 0/8 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:From c:\Users\urkow\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.

46/46 [==============================] - 1s 7ms/step - loss: 27.2973 - root_mean_squared_error: 5.2247 - val_loss: 19.1951 - val_root_mean_squared_error: 4.3812

Epoch 2/3                                            

46/46 [==============================] - 0s 2ms/step - loss: 13.9439 - root_mean_squared_error: 3.7342 - val_loss: 9.9222 - 

2026/05/26 12:35:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmpbbvhz9j0\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmpbbvhz9j0\model\data\model\assets



🏃 View run powerful-dolphin-261 at: http://localhost:5000/#/experiments/2/runs/ae098b5feb404eb2b8b7ce77a7845f74

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                     

46/46 [==============================] - 1s 4ms/step - loss: 31.5866 - root_mean_squared_error: 5.6202 - val_loss: 29.1455 - val_root_mean_squared_error: 5.3987

Epoch 2/3                                                                     

46/46 [==============================] - 0s 3ms/step - loss: 26.7991 - root_mean_squared_error: 5.1768 - val_loss: 24.7366 - val_root_mean_squared_error: 4.9736

Epoch 3/3                                                                     

46/46 [==============================] - 0s 2ms/step - loss: 22.7124 - root_mean_squared_error: 4.7658 - val_loss: 20.9603 - val_root_mean_squared_error: 4.5782

12/12 [==============================] - 0s 1ms/step - loss: 20.9603 - root_mean_squared_error: 4.57

...

🏃 View run skillful-crab-976 at: http://localhost:5000/#/experiments/2/runs/c73d9fd8d6cb4a4cbd27051ba27885e1

🧪 View experiment at: http://localhost:5000/#/experiments/2                 

100%|██████████| 8/8 [02:06<00:00, 15.77s/trial, best loss: 0.90754234790802]

2026/05/26 12:37:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmp5j3u48mb\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmp5j3u48mb\model\data\model\assets



Best parameters: {'lr': 0.003498671627867838, 'momentum': 0.8338979077718403}
Best eval rmse: 0.90754234790802
🏃 View run illustrious-mare-256 at: http://localhost:5000/#/experiments/2/runs/7f7c08df4411443dab5569cdc87e4952
🧪 View experiment at: http://localhost:5000/#/experiments/2


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [13]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

46/46 [==============================] - 2s 17ms/step - loss: 24.8996 - root_mean_squared_error: 4.9899 - val_loss: 18.2910 - val_root_mean_squared_error: 4.2768

Epoch 2/3                                            

46/46 [==============================] - 0s 9ms/step - loss: 13.8044 - root_mean_squared_error: 3.7154 - val_loss: 10.0528 - val_root_mean_squared_error: 3.1706

Epoch 3/3                                            

46/46 [==============================] - 0s 3ms/step - loss: 7.7080 - root_mean_squared_error: 2.7763 - val_loss: 5.8527 - val_root_mean_squared_error: 2.4192

12/12 [==============================] - 0s 12ms/step - loss: 5.8527 - root_mean_squared_error: 2.4192

  0%|          | 0/8 [00:03<?, ?trial/s, best loss=?]

2026/05/26 12:40:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmpluexpm_p\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmpluexpm_p\model\data\model\assets



🏃 View run beautiful-mole-215 at: http://localhost:5000/#/experiments/2/runs/3f96994dc6d444489d4f836f06a36468

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                     

46/46 [==============================] - 1s 5ms/step - loss: 12.1854 - root_mean_squared_error: 3.4908 - val_loss: 4.6394 - val_root_mean_squared_error: 2.1539

Epoch 2/3                                                                     

46/46 [==============================] - 0s 2ms/step - loss: 2.3714 - root_mean_squared_error: 1.5399 - val_loss: 1.6855 - val_root_mean_squared_error: 1.2982

Epoch 3/3                                                                     

46/46 [==============================] - 0s 3ms/step - loss: 1.3786 - root_mean_squared_error: 1.1742 - val_loss: 1.3219 - val_root_mean_squared_error: 1.1498

12/12 [==============================] - 0s 2ms/step - loss: 1.3219 - root_mean_squared_error: 1.1498

 12%

...

🏃 View run receptive-koi-241 at: http://localhost:5000/#/experiments/2/runs/9b7f4ac5a4674abeac435ad854728282

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [02:15<00:00, 16.88s/trial, best loss: 0.7510687708854675]

2026/05/26 12:42:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmp174yhvwe\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\urkow\AppData\Local\Temp\tmp174yhvwe\model\data\model\assets



Best parameters: {'lr': 0.00936975966734822, 'momentum': 0.8601498867029338}
Best eval rmse: 0.7510687708854675
🏃 View run unique-wren-721 at: http://localhost:5000/#/experiments/2/runs/6ecc9d890c6446f1a651afee1ae6ac8c
🧪 View experiment at: http://localhost:5000/#/experiments/2


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [14]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

ConnectionError: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001BDBC1C7650>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))

In [15]:
response.content

NameError: name 'response' is not defined